## Part 0 - Setup

Run this cell first, in every notebook. It fetches the course repository into the Colab session and moves into the `notebooks/` folder, so that the `../data/...` paths work.

It is safe to run more than once, and safe after a restart.

Note: outside Colab the cell does nothing except report the working directory. Start Jupyter from inside `notebooks/` and the paths work the same way.

If it prints `data ok: True`, you are set.


In [ ]:
# --- SETUP: run this first ---  [lares-setup-v1]
# works in Colab and locally, safe to run more than once
import os, sys, subprocess
REPO = "ai_bootcamp_foundations"
if "google.colab" in sys.modules:
    if not os.path.isdir(f"/content/{REPO}"):
        subprocess.run(["git", "clone", "-q",
                        f"https://github.com/unizg-fer-lares/{REPO}.git"],
                       cwd="/content", check=True)
    os.chdir(f"/content/{REPO}/notebooks")
print("cwd:", os.getcwd(), "| data ok:", os.path.isdir("../data"))


In [ ]:
import numpy as np, pandas as pd

# same plot styling as in 1a
import seaborn as sns
sns.set_theme(style="whitegrid")
import matplotlib.pyplot as plt
tex_fonts = {
    "font.family": "serif",
    # Use 26pt font in plots
    "axes.labelsize": 20,
    "font.size": 20,
    "figure.titlesize": 20,
    # Make the legend/label fonts a little smaller
    "legend.title_fontsize": 18,
    "legend.fontsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18
}

plt.rcParams.update(tex_fonts)


In [ ]:
# The sub-folder ``supervised_learning`` sits one level deeper than the
# shared ``data`` folder, so we add another ``../`` when needed.
DATA_DIR = "../data" if os.path.isdir("../data") else "../../data"
print("data folder:", DATA_DIR)


# Decision Trees and Support Vector Machines

**Student exercise - notebook `4_Decision_Trees_and_SVM`**

We train a decision tree on Iris petals, then compare SVM margins and kernels on small 2D datasets.


## Part 1 - Load the Iris data (petal features only)

Two features let us both fit the model and draw the boundary in one plot.


In [ ]:
X_train_full = pd.read_csv(f"{DATA_DIR}/iris/X_train.csv", index_col="Id")
X_test_full = pd.read_csv(f"{DATA_DIR}/iris/X_test.csv", index_col="Id")
y_train = pd.read_csv(f"{DATA_DIR}/iris/y_train.csv", index_col="Id").squeeze()
y_test = pd.read_csv(f"{DATA_DIR}/iris/y_test.csv", index_col="Id").squeeze()

feature_names = ["PetalLengthCm", "PetalWidthCm"]
X_train = X_train_full[feature_names]
X_test = X_test_full[feature_names]
species = {0: "Iris-setosa", 1: "Iris-versicolor", 2: "Iris-virginica"}

print("train flowers:", len(X_train), "| test flowers:", len(X_test))


## Part 2 - Decision trees: yes/no questions on the features

A decision tree asks a chain of threshold questions like *"is petal width < 0.8?"* to split the data into purer groups.

We first train a tree without any depth limit, then look at the tree itself.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, precision_score, recall_score

# TODO 1: create a DecisionTreeClassifier with no depth limit (use random_state=42)
#         and fit it on X_train, y_train.
unrestricted_tree = ...

print("tree depth :", unrestricted_tree.get_depth())
print("tree leaves:", unrestricted_tree.get_n_leaves())

### Draw the actual tree


In [ ]:
fig, ax = plt.subplots(figsize=(18, 9))
plot_tree(
    unrestricted_tree,
    feature_names=feature_names,
    class_names=list(species.values()),
    filled=True,
    rounded=True,
    ax=ax,
)
ax.set_title("Unrestricted decision tree")
plt.tight_layout()
plt.show()


## Part 3 - Prune the tree with max_depth=3

Setting `max_depth` limits how many questions the tree can ask in a row. This is a simple form of **pruning**: instead of splitting until every leaf is pure, we stop early.


In [ ]:
# TODO 2: create a DecisionTreeClassifier with max_depth=3 (random_state=42)
#         and fit it on X_train, y_train.
pruned_tree = ...

print("pruned depth :", pruned_tree.get_depth())
print("pruned leaves:", pruned_tree.get_n_leaves())

## Part 4 - Helper: draw decision boundaries

Same idea as in the KNN notebook. We cover the plot with a fine grid, ask the model to classify every point, and paint the background.


In [ ]:
def plot_decision_boundary(ax, model, X, y, title, show_svm_margin=False):
    # accept either a numpy array or a pandas DataFrame
    if isinstance(X, pd.DataFrame):
        X_values = X.values
        columns = list(X.columns)
    else:
        X_values = np.asarray(X)
        columns = None

    x_min = X_values[:, 0].min() - 0.5
    x_max = X_values[:, 0].max() + 0.5
    y_min = X_values[:, 1].min() - 0.5
    y_max = X_values[:, 1].max() + 0.5

    grid_x, grid_y = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300),
    )
    grid_points = np.c_[grid_x.ravel(), grid_y.ravel()]
    # feed the grid back as a DataFrame when the model was fitted with column names
    if columns is not None:
        grid_input = pd.DataFrame(grid_points, columns=columns)
    else:
        grid_input = grid_points
    grid_pred = model.predict(grid_input).reshape(grid_x.shape)

    ax.contourf(grid_x, grid_y, grid_pred, alpha=0.22, cmap="coolwarm")
    if show_svm_margin and hasattr(model, "decision_function"):
        scores = model.decision_function(grid_input).reshape(grid_x.shape)
        ax.contour(grid_x, grid_y, scores, levels=[-1, 0, 1],
                   colors=["gray", "black", "gray"], linestyles=["--", "-", "--"])

    ax.scatter(X_values[:, 0], X_values[:, 1], c=y, cmap="coolwarm", edgecolor="white", s=55)
    ax.set_title(title)


### Boundaries of the two trees

The unrestricted tree can carve out small islands around every group of training points. The depth-3 tree paints broader regions.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)
plot_decision_boundary(axes[0], unrestricted_tree, X_train, y_train, "Unrestricted tree")
plot_decision_boundary(axes[1], pruned_tree, X_train, y_train, "Pruned tree (max_depth=3)")
for ax in axes:
    ax.set_xlabel("Petal length (cm)")
axes[0].set_ylabel("Petal width (cm)")
plt.tight_layout()
plt.show()

tree_rows = []
for label, model in {"Unrestricted": unrestricted_tree, "Pruned": pruned_tree}.items():
    test_pred = model.predict(X_test)
    tree_rows.append({
        "model": label,
        "accuracy": accuracy_score(y_test, test_pred),
        "precision": precision_score(y_test, test_pred, average="weighted"),
        "recall": recall_score(y_test, test_pred, average="weighted"),
    })
pd.DataFrame(tree_rows).set_index("model").round(3)

## Part 5 - Support Vector Machines

An SVM draws a straight line (or a curve) that separates the two classes with the **widest possible gap** between them. The gap is called the **margin** and the points touching it are the **support vectors**.

The parameter `C` controls how much we punish mistakes:

* **High C** - almost no mistakes are allowed, so the boundary shifts to chase outliers (hard margin).
* **Low C** - a wider gap, at the cost of a few points landing on the wrong side (soft margin).

To make the effect visible we build a small 2D dataset by hand: two clusters with two annoying outliers.


In [ ]:
rng = np.random.default_rng(4)
cluster_a = rng.normal(loc=(-1.5, -1.0), scale=0.6, size=(60, 2))
cluster_b = rng.normal(loc=(1.5, 1.0), scale=0.6, size=(60, 2))
outliers = np.array([[1.6, -1.1], [-1.5, 1.3]])  # two awkward points near the wrong cluster

X_margin = np.vstack([cluster_a, cluster_b, outliers])
y_margin = np.array([0] * 60 + [1] * 60 + [0, 1])

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_margin[:, 0], X_margin[:, 1], c=y_margin, cmap="coolwarm", edgecolor="white", s=55)
ax.set_title("Two clusters plus two awkward points")
plt.tight_layout()
plt.show()


### Hard margin (high C) vs. soft margin (low C)

Dashed lines mark the margin; the solid line is the decision boundary.


In [ ]:
from sklearn.svm import SVC

# TODO 3: fit two linear-kernel SVMs on (X_margin, y_margin):
#   - svm_hard uses C=100 (hard margin)
#   - svm_soft uses C=0.05 (soft margin)
svm_hard = ...
svm_soft = ...

# Plotting the margins and support vectors is prepared for you.
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)
plot_decision_boundary(axes[0], svm_hard, X_margin, y_margin, "Hard margin (C=100)", show_svm_margin=True)
plot_decision_boundary(axes[1], svm_soft, X_margin, y_margin, "Soft margin (C=0.05)", show_svm_margin=True)

for ax, model in zip(axes, [svm_hard, svm_soft]):
    supports = model.support_vectors_
    ax.scatter(supports[:, 0], supports[:, 1], s=180, facecolors="none",
               edgecolors="black", linewidths=2, label="support vectors")
    ax.legend(fontsize=13)

plt.tight_layout()
plt.show()

## Part 6 - The kernel trick on circles

Some classes cannot be separated by any straight line. A **kernel** lets an SVM behave as if it worked in a richer feature space without us having to compute new features by hand.

We build two concentric rings by hand to demonstrate:

* the linear kernel cannot separate them,
* the RBF kernel can wrap around the inner ring.


In [ ]:
rng = np.random.default_rng(0)
n_per_ring = 150

angle_inner = rng.uniform(0, 2 * np.pi, n_per_ring)
inner_ring = np.column_stack([np.cos(angle_inner), np.sin(angle_inner)]) * 0.4
inner_ring += rng.normal(scale=0.05, size=inner_ring.shape)

angle_outer = rng.uniform(0, 2 * np.pi, n_per_ring)
outer_ring = np.column_stack([np.cos(angle_outer), np.sin(angle_outer)]) * 1.0
outer_ring += rng.normal(scale=0.05, size=outer_ring.shape)

X_circles = np.vstack([inner_ring, outer_ring])
y_circles = np.array([0] * n_per_ring + [1] * n_per_ring)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap="coolwarm", edgecolor="white", s=45)
ax.set_title("Two rings - no straight line can separate them")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()


### Linear kernel vs. RBF kernel


In [ ]:
# TODO 4: fit two SVMs on (X_circles, y_circles):
#   - linear_svm uses kernel="linear" with C=1
#   - rbf_svm uses kernel="rbf" with C=1 and gamma="scale"
linear_svm = ...
rbf_svm = ...

# Plotting and metric computation are prepared for you.
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=True, sharey=True)
plot_decision_boundary(axes[0], linear_svm, X_circles, y_circles, "Linear kernel (fails)")
plot_decision_boundary(axes[1], rbf_svm, X_circles, y_circles, "RBF kernel (wraps inner ring)")
for ax in axes:
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

svm_rows = []
for label, model in {"Linear kernel": linear_svm, "RBF kernel": rbf_svm}.items():
    pred = model.predict(X_circles)
    svm_rows.append({
        "model": label,
        "accuracy": accuracy_score(y_circles, pred),
        "precision": precision_score(y_circles, pred),
        "recall": recall_score(y_circles, pred),
    })
pd.DataFrame(svm_rows).set_index("model").round(3)

### Conclusions - overall:
*   Decision trees ask yes/no questions until each leaf is pure; an unrestricted tree can memorise noise.
*   Setting max_depth prunes the tree and gives cleaner, more general decision regions.
*   The SVM parameter C balances the width of the margin against how many mistakes are tolerated.
*   The choice of kernel decides what shapes an SVM can separate; a linear kernel fails on rings while an RBF kernel wraps around them.

_University of Zagreb Faculty of Electrical Engineering and Computing_  
_Laboratory for Renewable Energy Systems_  

_Course: AI bootcamp - basic_  
_Notebook: 4_Decision_Trees_and_SVM_  

_Website: [www.lares.fer.hr](https://www.lares.fer.hr/)_  
_Contact: [Hrvoje Novak](mailto:hrvoje.novak@fer.hr)_
